## Project: Mapping the Potential Destructive Power of Wildfires Using Machine Learning

### Module 5: *Evaluation and Visualization*

---
### Contents  
- 1. Generate Case Study Predictions
- 2. Geospatial Visualization
---
### Notes
This section applies the trained models to predict wildfire severity in two real-world scenarios:

#### Palisades Fire (January 2025)  
#### Dixie Fire (July 2021)

We evaluate and compare the performance of three classification models:

- **XGBoost** (multi-class classification)
- **Random Forest**
- **K-Nearest Neighbors (KNN)**
---
### Inputs

---
### Outputs  

---
### User Created Dependencies  

In [1]:
# Add the parent directory to the system path so "src" can be found
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..')))

# user built utilities
from src.plot_utils import plot_map
from src.plot_utils import individual_plot_map
from src.plot_utils import interpolate_idw
from src.plot_utils import interpolate_idw2

---
### Third Party Dependencies

In [2]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Modeling libraries
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import RidgeClassifier
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.model_selection import train_test_split

# Geospatial libraries
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.lines as mlines  # For custom map legends

from datetime import timedelta

In [3]:
# Features and labels
X = pd.read_csv('../data/processed/X.csv')
y = pd.read_csv('../data/processed/y.csv').squeeze()  # Load as Series
details = pd.read_csv('../data/processed/details.csv')


### 1.1 Build Models

In [4]:
model_parameters = pd.read_csv('../data/processed/model_parameters.csv', index_col=0)

RF_parameters = model_parameters.loc['RandomForest'].dropna().to_dict()
XGB_parameters = model_parameters.loc['XGBoost'].dropna().to_dict()
Knn_parameters = model_parameters.loc['KNN'].dropna().to_dict()
optimal_learning_rate = XGB_parameters['learning_rate']

# Helper function to convert to int if possible
def convert_to_int(d):
    return {k: int(float(v)) if str(v).replace('.', '', 1).isdigit() else v for k, v in d.items()}

RF_parameters = convert_to_int(RF_parameters)
XGB_parameters = convert_to_int(XGB_parameters)
Knn_parameters = convert_to_int(Knn_parameters)

XGB_parameters['learning_rate'] = optimal_learning_rate

In [5]:
optimal_learning_rate = XGB_parameters['learning_rate']

In [6]:
# Helper function to convert to int if possible
def convert_to_int(d):
    return {k: int(float(v)) if str(v).replace('.', '', 1).isdigit() else v for k, v in d.items()}

RF_parameters = convert_to_int(RF_parameters)
XGB_parameters = convert_to_int(XGB_parameters)
Knn_parameters = convert_to_int(Knn_parameters)


In [7]:
XGB_parameters['learning_rate'] = optimal_learning_rate

In [8]:
display(RF_parameters)
display(XGB_parameters)
display(Knn_parameters)

{'n_estimators': 150,
 'max_depth': 20,
 'min_samples_split': 10,
 'max_features': 'sqrt',
 'class_weight': 'balanced'}

{'n_estimators': 50,
 'max_depth': 6,
 'objective': 'multi:softmax',
 'num_class': 3,
 'learning_rate': 0.4}

{'n_neighbors': 4, 'weights': 'uniform'}

In [9]:
# Final tuned models
optimum_xgb_model = xgb.XGBClassifier(**XGB_parameters)
optimum_rf = RandomForestClassifier(**RF_parameters)
optimum_knn = KNeighborsClassifier(**Knn_parameters)

### 1.2 Train Models

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [11]:
optimum_xgb_model.fit(X_train, y_train)
optimum_rf.fit(X_train, y_train)
optimum_knn.fit(X_train, y_train)

KNeighborsClassifier(n_neighbors=4)

In [12]:
rf_importances = optimum_rf.feature_importances_
for name, importance in zip(X.columns, rf_importances):
    print(f"{name}: {importance:.4f}")

Avg Air Temp (F): 0.0942
Avg Rel Hum (%): 0.1063
Avg Wind Speed (mph): 0.1131
Avg Soil Temp (F): 0.0876
Precip (in): 0.0181
Avg Vap Pres (mBars): 0.0833
Sol Rad (Ly/day): 0.1636
ETo (in): 0.1137
Total Population: 0.0769
density: 0.0820
Mean Income: 0.0611


In [13]:
xgb_importances = optimum_xgb_model.feature_importances_
for name, importance in zip(X.columns, xgb_importances):
    print(f"{name}: {importance:.4f}")

Avg Air Temp (F): 0.0827
Avg Rel Hum (%): 0.0849
Avg Wind Speed (mph): 0.0796
Avg Soil Temp (F): 0.0700
Precip (in): 0.1159
Avg Vap Pres (mBars): 0.0527
Sol Rad (Ly/day): 0.1294
ETo (in): 0.0797
Total Population: 0.1346
density: 0.0962
Mean Income: 0.0742


In [14]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

In [15]:
def evaluate_model(model, X_test, y_test, name="Model"):
    y_pred = model.predict(X_test)
    print(f"Evaluation for {name}")
    print(f"Accuracy:  {accuracy_score(y_test, y_pred):.3f}")
    print(f"Precision: {precision_score(y_test, y_pred, average='weighted'):.3f}")
    print(f"Recall:    {recall_score(y_test, y_pred, average='weighted'):.3f}")
    print(f"F1 Score:  {f1_score(y_test, y_pred, average='weighted'):.3f}")
    print("---- Classification Report ----")
    print(classification_report(y_test, y_pred))
    print("\n")

In [16]:
evaluate_model(optimum_xgb_model, X_test, y_test, "XGBoost")
evaluate_model(optimum_rf, X_test, y_test, "Random Forest")
evaluate_model(optimum_knn, X_test, y_test, "KNN")

Evaluation for XGBoost
Accuracy:  0.400
Precision: 0.410
Recall:    0.400
F1 Score:  0.405
---- Classification Report ----
              precision    recall  f1-score   support

           0       0.50      0.50      0.50        14
           1       0.10      0.11      0.11         9
           2       0.50      0.47      0.48        17

    accuracy                           0.40        40
   macro avg       0.37      0.36      0.36        40
weighted avg       0.41      0.40      0.40        40



Evaluation for Random Forest
Accuracy:  0.475
Precision: 0.438
Recall:    0.475
F1 Score:  0.453
---- Classification Report ----
              precision    recall  f1-score   support

           0       0.62      0.57      0.59        14
           1       0.00      0.00      0.00         9
           2       0.52      0.65      0.58        17

    accuracy                           0.47        40
   macro avg       0.38      0.41      0.39        40
weighted avg       0.44      0.47      

### 1.3 Generate Predictions

In [17]:
# Create geometry from lat/lon
pal_geometry = [Point(xy) for xy in zip(frank_details['Longitude'], frank_details['Latitude'])]
dixie_geometry = [Point(xy) for xy in zip(mount_details['Longitude'], mount_details['Latitude'])]

# Convert to GeoDataFrames
pal_gdf = gpd.GeoDataFrame(frank_details, geometry=pal_geometry, crs="EPSG:4326")
dixie_gdf = gpd.GeoDataFrame(mount_details, geometry=dixie_geometry, crs="EPSG:4326")

NameError: name 'frank_details' is not defined

## 2 Geospatial Visualization of Model Predictions

This section visualizes wildfire severity predictions made by each model on specific dates during the Palisades Fire (Jan 2025) and the Dixie Fire (July 2021).

### 2.1 Palisades + Eaton Fire Predictions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))  # 1 row, 3 columns

# Plot into each subplot by passing in the axis
plot_map(pal_gdf, 'Prediction RF', 'Franklin', ax=axes[0])
plot_map(pal_gdf, 'Prediction KNN', 'Franklin', ax=axes[1])
plot_map(pal_gdf, 'Prediction XGB', 'Franklin', ax=axes[2])

# Set titles if plot_map doesn't do it
axes[0].set_title('Southern California Random Forest 01/07/2025')
axes[1].set_title('Southern California KNN 01/07/2025')
axes[2].set_title('Southern California XGB 01/07/2025')

plt.tight_layout()

plt.savefig("../plots/Palisades_predictions.png", dpi=300)

### 2.2 Dixie Fire Predictions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))  # 1 row, 3 columns

# Plot into each subplot by passing in the axis
plot_map(dixie_gdf, 'Prediction RF', 'Mountain', ax=axes[0])
plot_map(dixie_gdf, 'Prediction KNN', 'Mountain', ax=axes[1])
plot_map(dixie_gdf, 'Prediction XGB', 'Mountain', ax=axes[2])

# Set titles if plot_map doesn't do it
axes[0].set_title('Random Forest')
axes[1].set_title('KNN')
axes[2].set_title('XGBoost')

plt.tight_layout()
plt.show()

### 2.3 California

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))  # 1 row, 3 columns

# Plot into each subplot by passing in the axis
plot_map(pal_gdf, 'Prediction RF', 'Franklin', ax=axes[0], cali = True)
plot_map(dixie_gdf, 'Prediction RF', 'Mountain', ax=axes[1], cali = True)

# Set titles if plot_map doesn't do it
axes[0].set_title('RF Palisades')
axes[1].set_title('RF Dixie')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))  # 1 row, 3 columns

# Plot into each subplot by passing in the axis
plot_map(pal_gdf, 'Prediction KNN', 'Franklin', ax=axes[0], cali = True)
plot_map(dixie_gdf, 'Prediction KNN', 'Mountain', ax=axes[1], cali = True)

# Set titles if plot_map doesn't do it
axes[0].set_title('KNN Palisades')
axes[1].set_title('KNN Dixie')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))  # 1 row, 3 columns

# Plot into each subplot by passing in the axis
plot_map(pal_gdf, 'Prediction XGB', 'Franklin', ax=axes[0], cali = True)
plot_map(dixie_gdf, 'Prediction XGB', 'Mountain', ax=axes[1], cali = True)

# Set titles if plot_map doesn't do it
axes[0].set_title('XGB Palisades')
axes[1].set_title('XGB Dixie')

plt.tight_layout()
plt.show()

In [ ]:
c_map = {'Low':0,'Moderate':1,'High':2} 
pred_cols = ['Prediction KNN','Prediction RF','Prediction XGB']
loc_cols = ['Pal','Dixie']

In [ ]:
for col in pred_cols:
    pal_gdf[col] = pal_gdf[col].map(c_map)
    
for col in pred_cols:
    dixie_gdf[col] = dixie_gdf[col].map(c_map)

In [ ]:
pal_gdf['firename'] = 'Pal'
dixie_gdf['firename'] = 'Dixie'
combined_gdf = gpd.GeoDataFrame(pd.concat([pal_gdf, dixie_gdf], ignore_index=True), crs=pal_gdf.crs)
gdf = combined_gdf[combined_gdf['firename'] == 'Pal']


In [ ]:
fire_points = [
    {
        'name': 'Park Fire',
        'lat': 39.7789,
        'lon': -121.76168,
        'color': 'gold'
    }
]


interpolated = interpolate_idw2(gdf_points = gdf, value_column = 'Prediction XGB', fire_points = fire_points,
                                k = 5, grid_spacing = 0.05,
                                buffer = 0.1,
                                title = 'Southern California Interpolated Prediction XGB - 12/09/2024',
                                plot = True)

In [ ]:
unscaled_X['Date'] = pd.to_datetime(unscaled_X['Date']).dt.date

In [ ]:
frank = unscaled_X[unscaled_X['Date']==FRANKLIN_FIRE]

In [ ]:
frank['Target'].value_counts()

In [ ]:
frank

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# 1. Make sure the inputs are aligned
y_true = frank['Target'].values
y_pred = pal_gdf['Prediction RF'].values

# 2. Generate confusion matrix
cm = confusion_matrix(y_true, y_pred)

# 3. Display it
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()